In [ ]:
from functools import partial
from typing import List, Tuple

import numpy as np
from ipywidgets import widgets, HBox, VBox, Layout
from IPython.display import HTML, display, update_display

import math

In [ ]:
class Colors:
    START = 'green'
    GOAL = 'blue'
    OBSTACLE = 'black'
    PATH = 'red'
    VISITED = 'yellow'


class Interface:
    def __init__(self, shape=(10,10)):
        self.n_rows, self.n_cols = shape
        self.buttons = [
            [
                widgets.Button(
                    description='',
                    layout=Layout(width='50px', height='50px')
                )
                for _ in range(self.n_cols)
            ]
            for _ in range(self.n_rows)
        ]

        self.map_widget = VBox([HBox(row) for row in self.buttons])
        self.link_positions()

        self.start_pos, self.goal_pos = None, None
        self.obstacles = set()

    def on_select_position(self, pos: tuple, button: widgets.Button):
        """Callback on user click on a position of the map

        :param pos: row, column of button on grid
        :param button: clicked button
        """
        if self.start_pos is None:
            self.start_pos = pos
            button.style.button_color = Colors.START

        elif self.goal_pos is None:
            self.goal_pos = pos
            button.style.button_color = Colors.GOAL

        else:
            self.obstacles.add(pos)
            button.style.button_color = Colors.OBSTACLE

    def link_positions(self):
        """Link clicks on buttons"""
        for i, row in enumerate(self.buttons):
            for j, button in enumerate(row):
                button.on_click(partial(self.on_select_position, (i, j)))

    def disable_buttons(self):
        """Disable buttons to avoid clicks after end game"""
        for row in self.buttons:
            for button in row:
                button.disabled = True

    def start(self):
        """Display map interface"""
        display(self.map_widget)



In [ ]:
interface = Interface()
interface.start()

In [ ]:
MOVES = [
    (+1, 0),
    (-1, 0),
    (0, +1),
    (0, -1),
    (+1, +1),
    (-1, -1),
    (+1, -1),
    (-1, +1),
] # MOVES = quais são os movimentos possíveis


def BFS(start_pos: Tuple[int, int], goal_pos: Tuple[int, int],
        obstacles: List[Tuple[int, int]], max_y: int, max_x: int):



    """Breadth-first search algorithm"""

    def is_valid(position: Tuple[int, int]):
        """Checks if a position is valid
        i.e. it is inside map and it is not a obstacle"""
        y, x = position
        return 0 <= x < max_x and 0 <= y < max_y and position not in obstacles

    def get_next_positions(y, x):
        """Get next possible moves"""
        positions = []
        for dy, dx in MOVES:
            new_pos = (y + dy + x + dx)
            if is_valid(new_pos):
                positions.append(new_pos)
        return positions

    # crie uma lista de n'os a visitar e já visitados
    a_visitar = [start_pos]
    ja_visitados = []
    caminhos = {
        start_pos : [start_pos]
    }
    f = {start_pos: + heuristics(start_pos)}
    g = {start_pos: 0}

    while a_visitar:
        current = a_visitar.pop(0)
        if current in visited:
            continue

        # verifica se é o objeto
        if current in visited == goal_pos:
            return path[goal_pos], visited

        visited.append(current)

        for pos in where_can_i_go(*current):
            a_visitar.append(pos)
            path[pos] = path[current] + [pos]

    return [], visited


In [ ]:
MOVES_A_STAR = [
    (+1, 0),
    (-1, 0),
    (0, +1),
    (0, -1),
    (+1, +1),
    (-1, -1),
    (+1, -1),
    (-1, +1),
]  # quais são os movimentos possíveis (adicione diagonais)


def a_star(start_pos: Tuple[int, int], goal_pos: Tuple[int, int],
        obstacles: List[Tuple[int, int]], max_y: int, max_x: int):
    """Breadth-first search algorithm"""

    def heuristics(position: Tuple[int, int]):
       return math.sqrt((goal_pos[0] - position[0]) ** 2 + (goal_pos[1] - position[1]) ** 2)

    def is_valid(position: Tuple[int, int]):
        """Checks if a position is valid
        i.e. it is inside map and it is not a obstacle"""
        y, x = position
        return 0 <= x < max_x and 0 <= y < max_y and position not in obstacles


    def get_next_positions(y, x):
        """Get next possible moves and it cost"""
        for dy, dx in MOVES:
            new_pos = (y + dy, x + dx)
            if is_valid(new_pos):
                cost = math.sqrt(dy ** 2 + dx ** 2)
                positions.append((new_pos, cost))
        return positions

    to_visit = [start_pos]
    visited = []
    path = {start_pos: (start_pos)}
    f = {start_pos: 0 + heuristics(start_pos)}
    g = {start_pos: 0}

    while to_visit:
        current = min(to_visit, key=lambda x: f[x])
        to_visit.remove(current)

        if current == goal_pos:
            return path[goal_pos], visited

        visited.append(current)

        for pos, cost in get_next_positions(*current):
            if pos not in g or g[current] + cost < g[pos]:
                g[pos] = g[current] + cost
                f[pos] = g[pos] + heuristics(pos)

                path[pos] = path[current] + [pos]
                if pos not in to_visit:
                    to_visit.append(pos)
    return [], visited


In [ ]:
class PathFinder(Interface):
    def __init__(self, shape=(10, 10)):
        super().__init__(shape)
        self.find_button = widgets.Button(
            description='Find shortest path A*',
            style=widgets.ButtonStyle(button_color=Colors.START)
        )

        self.find_button_bfs = widgets.Button(
            description='BFS',
            style=widgets.ButtonStyle(button_color=Colors.START)
        )

        self.clear_button = widgets.Button(
            description='clear',
            style=widgets.ButtonStyle(button_color=Colors.VISITED)
        )

        self.find_button.on_click(self.search)
        self.find_button_bfs.on_click(self.search_bfs)
        self.clear_button.on_click(self.clear)

    def clear(self, *args):
        for row in self.buttons:
            for cell in row:
                if cell.style.button_color in [Colors.VISITED, Colors.PATH]:
                    cell.style.button_color = None

    def set_path(self, path: List[Tuple[int, int]]):
        """Color a path in map"""
        for y, x in path[1:]:
            self.buttons[y][x].style.button_color = Colors.PATH

    def set_visited(self, visited: List[Tuple[int, int]]):
        """Color a path in map"""
        for y, x in visited:
            if self.buttons[y][x].style.button_color not in {Colors.START, Colors.GOAL, Colors.OBSTACLE, Colors.PATH}:
                self.buttons[y][x].style.button_color = Colors.VISITED

    def search(self, *args):
        """Disable editions on map, then finds the shorterst path and draw it"""
        self.disable_buttons()
        path, visited_positions = a_star(self.start_pos, self.goal_pos, self.obstacles, self.n_rows, self.n_cols)
        print(len(visited_positions))
        self.set_visited(visited_positions)
        self.set_path(path)
        return path

    def search_bfs(self, *args):
        """Disable editions on map, then finds the shorterst path and draw it"""
        self.disable_buttons()
        path, visited_positions = BFS(self.start_pos, self.goal_pos, self.obstacles, self.n_rows, self.n_cols)
        print(len(visited_positions))
        self.set_visited(visited_positions)
        self.set_path(path)
        return path

    def start(self):
        """Display map interface"""
        display(VBox(
            [
                self.map_widget, 
                self.find_button, 
                self.find_button_bfs, 
                self.clear_button
                ]
            ))



In [ ]:
path_finder = PathFinder()
path_finder.start()